# 02 - Fund Flow Analysis
## Polymarket Obfuscation Detection Pipeline

This notebook analyzes the fund flow patterns and traces the origin of funds for suspicious wallets.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

from src.storage.postgres_store import PostgresStore
from src.data.onchain_client import OnchainClient, MIXER_CONTRACTS, BRIDGE_CONTRACTS

## Initialize Connections

In [ ]:
store = PostgresStore()
onchain = OnchainClient()

print("Connected to database and on-chain client")

## Get Flagged Wallets

In [ ]:
wallets = store.get_all_wallets()
flagged_wallets = [w for w in wallets if w.risk_score > 25]
print(f"Flagged wallets: {len(flagged_wallets)}")

## Trace Fund Origins

In [ ]:
def analyze_fund_flow(address):
    hops = onchain.trace_fund_origin(address, max_hops=5)
    
    contract_types = defaultdict(int)
    total_amount = 0
    
    for hop in hops:
        contract_types[hop['contract_type']] += 1
        try:
            total_amount += float(hop['amount'])
        except:
            pass
    
    return {
        'address': address,
        'hop_count': len(hops),
        'total_amount': total_amount,
        'contract_types': dict(contract_types),
        'has_mixer': any('mixer' in ct for ct in contract_types),
        'has_bridge': any('bridge' in ct for ct in contract_types)
    }

print("Tracing fund origins for flagged wallets...")

In [ ]:
flow_analysis = []
for wallet in flagged_wallets[:20]:
    try:
        analysis = analyze_fund_flow(wallet.address)
        flow_analysis.append(analysis)
        print(f"Traced {wallet.address[:10]}... - {analysis['hop_count']} hops")
    except Exception as e:
        print(f"Error tracing {wallet.address[:10]}...: {e}")

In [ ]:
df_flow = pd.DataFrame(flow_analysis)
df_flow.head(10)

## Mixer/Bridge Usage Statistics

In [ ]:
mixer_count = df_flow['has_mixer'].sum()
bridge_count = df_flow['has_bridge'].sum()

print(f"Wallets with Mixer Interaction: {mixer_count} ({mixer_count/len(df_flow)*100:.1f}%)")
print(f"Wallets with Bridge Interaction: {bridge_count} ({bridge_count/len(df_flow)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

df_flow['hop_count'].hist(bins=20, ax=axes[0])
axes[0].set_title('Distribution of Fund Flow Hops')
axes[0].set_xlabel('Number of Hops')

df_flow['total_amount'].hist(bins=20, ax=axes[1])
axes[1].set_title('Distribution of Total Fund Amounts')
axes[1].set_xlabel('Total Amount (USD)')

suspect_types = pd.DataFrame({
    'Type': ['Has Mixer', 'Has Bridge'],
    'Count': [mixer_count, bridge_count]
})
suspect_types.set_index('Type').plot(kind='bar', ax=axes[2], color=['red', 'orange'])
axes[2].set_title('Suspicious Activity Indicators')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

## Contract Type Distribution

In [ ]:
all_contract_types = defaultdict(int)
for ct_dict in df_flow['contract_types']:
    for ct, count in ct_dict.items():
        all_contract_types[ct] += count

contract_df = pd.DataFrame({
    'Contract Type': list(all_contract_types.keys()),
    'Count': list(all_contract_types.values())
}).sort_values('Count', ascending=False)

contract_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
contract_df.set_index('Contract Type').plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Contract Type Distribution in Fund Flows')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

## Detailed Wallet Analysis

In [ ]:
high_risk_wallets = df_flow[df_flow['has_mixer'] | (df_flow['hop_count'] > 3)].sort_values('hop_count', ascending=False)
high_risk_wallets

## Save Analysis Results

In [ ]:
df_flow.to_csv('../reports/fund_flow_analysis.csv', index=False)
print("Fund flow analysis saved to ../reports/fund_flow_analysis.csv")

In [ ]:
store.close()
print("Connections closed")